# Confirmatory specialization experiment

Leakage-safe CIFAR-100 + slimmable ResNet-18 experiment: three shared models, twelve validation-selected specialized models, fixed train-derived BN calibration, pre-specialization Wasserstein geometry, and leave-one-seed-out predictor comparison. Run all cells once with Kaggle Internet, a 2xT4 accelerator when available, and the `github_token` secret.

## 1. Environment and secure clone

In [ ]:
import os, subprocess, sys
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU before continuing'
print('Detected GPUs:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()): print(f'GPU {i}: {torch.cuda.get_device_name(i)}')
print('CUDA:', torch.version.cuda, 'PyTorch:', torch.__version__)

In [ ]:
from kaggle_secrets import UserSecretsClient
try:
    github_token = UserSecretsClient().get_secret('github_token')
except Exception as exc:
    raise RuntimeError("Attach a Kaggle secret named 'github_token'.") from exc
assert github_token
REPO_URL = 'https://github.com/duyh80456-code/new-pruning.git'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
git_env = os.environ.copy()
git_env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN_RUNTIME': github_token})
try:
    command = ['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'] if (PROJECT_ROOT / '.git').is_dir() else ['git', 'clone', REPO_URL, str(PROJECT_ROOT)]
    subprocess.run(command, env=git_env, check=True)
finally:
    askpass.unlink(missing_ok=True)
    git_env.pop('GITHUB_TOKEN_RUNTIME', None)
    github_token = None
assert (PROJECT_ROOT / 'configs' / 'kaggle_confirmatory_specialization.yaml').is_file(), 'Commit and push the confirmatory files first'
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'thop>=0.1.1', 'tabulate>=0.9'], check=True)

## 2. Lock and verify the protocol

In [ ]:
from datetime import datetime, timezone
import yaml
BASE_CONFIG = PROJECT_ROOT / 'configs' / 'kaggle_confirmatory_specialization.yaml'
config = yaml.safe_load(BASE_CONFIG.read_text())
assert config['dataset']['name'].lower() == 'cifar100' and config['dataset']['fake_data'] is False
assert config['dataset']['validation_size'] == 5000 and config['dataset']['bn_calibration_size'] == 6400
assert config['dataset']['feature_subset_size'] == 2000
assert config['model']['backbone'] == 'slimmable_resnet18'
assert config['experiment']['seeds'] == [0, 1, 2]
assert config['training']['epochs'] == 20 and config['training']['width_loss_reduction'] == 'mean'
assert config['compression']['train_widths'] == [0.25, 0.50, 0.75, 1.00]
assert config['compression']['eval_widths'] == [round(0.25 + 0.05*i, 2) for i in range(16)]
assert config['specialization'] == {'widths': [0.30, 0.40, 0.60, 0.80], 'epochs': 30, 'learning_rate': 0.01}
RUN_NAME = datetime.now(timezone.utc).strftime('kaggle-confirmatory-%Y%m%d-%H%M%S')
RUN_DIR = Path('/kaggle/working/new-pruning-outputs') / RUN_NAME
config['experiment']['output_dir'] = str(RUN_DIR)
RESOLVED_CONFIG = Path('/kaggle/working/kaggle_confirmatory_resolved.yaml')
RESOLVED_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
print(yaml.safe_dump(config, sort_keys=False))
print('Expected wall time: roughly 3.5-5.5 hours on 2xT4, or 6-8 hours on 1xT4.')

In [ ]:
from data import build_confirmatory_loaders
loaders0 = build_confirmatory_loaders(config, training_seed=0)
loaders1 = build_confirmatory_loaders(config, training_seed=1)
assert len(loaders0.train.dataset) == 45_000
assert len(loaders0.validation.dataset) == 5_000
assert len(loaders0.test.dataset) == 10_000
assert len(loaders0.calibration.dataset) == 6_400
assert len(loaders0.geometry.dataset) == 2_000
cal_ids0 = [int(x) for _, _, ids in loaders0.calibration for x in ids]
cal_ids1 = [int(x) for _, _, ids in loaders1.calibration for x in ids]
geo_ids0 = [int(x) for _, _, ids in loaders0.geometry for x in ids]
geo_ids1 = [int(x) for _, _, ids in loaders1.geometry for x in ids]
assert cal_ids0 == cal_ids1 and geo_ids0 == geo_ids1
assert set(cal_ids0).issubset(set(loaders0.train.dataset.indices))
assert set(geo_ids0).issubset(set(loaders0.validation.dataset.indices))
assert set(cal_ids0).isdisjoint(set(loaders0.validation.dataset.indices))
del loaders0, loaders1, cal_ids0, cal_ids1, geo_ids0, geo_ids1
print('45k/5k/test split and fixed calibration/geometry samples: OK')

## 3. Launch three shared and twelve specialized runs

In [ ]:
import time
from scripts.run_confirmatory import run, finalize_confirmatory
from scripts.run_multi_gpu import run_multi_gpu
started = time.perf_counter()
if torch.cuda.device_count() >= 2:
    print('Using two-GPU dynamic seed scheduler.')
    report_path = run_multi_gpu(RESOLVED_CONFIG, [0, 1], runner_module='scripts.run_confirmatory', finalize=finalize_confirmatory)
else:
    print('Only one GPU detected; using sequential fallback.')
    report_path = run(RESOLVED_CONFIG)
elapsed_hours = (time.perf_counter() - started) / 3600
print(f'Completed in {elapsed_hours:.2f} hours')
print('Base report:', report_path)

## 4. Integrity checks

In [ ]:
import json, numpy as np, pandas as pd
for seed in [0, 1, 2]:
    seed_dir = RUN_DIR / f'seed_{seed}'
    training = pd.read_csv(seed_dir / 'training_metrics.csv')
    budget = pd.read_csv(seed_dir / 'results' / 'budget_metrics.csv')
    selected = pd.read_csv(seed_dir / 'results' / 'selected_specialized_checkpoints.csv')
    specialized = pd.read_csv(seed_dir / 'results' / 'specialized_metrics.csv')
    assert len(training) == 80 and set(training['width']) == {0.25, 0.50, 0.75, 1.00}
    assert len(budget) == 16 and np.isfinite(budget[['accuracy', 'loss', 'flops', 'params']]).all().all()
    assert len(selected) == 4 and set(selected['width'].round(2)) == {0.30, 0.40, 0.60, 0.80}
    assert len(specialized) == 4 and np.isfinite(specialized[['shared_test_accuracy', 'specialized_test_accuracy', 'specialization_gap']]).all().all()
    assert selected['epoch'].between(0, 30).all()
    assert len(list((seed_dir / 'specialized').glob('best_budget_*.pt'))) == 4
    histories = sorted((seed_dir / 'results').glob('specialization_history_budget_*.csv'))
    assert len(histories) == 4
    for path in histories:
        history = pd.read_csv(path)
        assert len(history) == 31 and history['epoch'].tolist() == list(range(31))
print('Three shared runs, twelve best checkpoints, and final test metrics: OK')

## 5. Dense-grid G-versus-P stability

In [ ]:
from scipy.stats import pearsonr, spearmanr
correlation_rows, local_frames = [], []
for seed in [0, 1, 2]:
    seed_dir = RUN_DIR / f'seed_{seed}'
    local = pd.read_csv(seed_dir / 'results' / 'local_sensitivity.csv')
    local['seed'] = seed
    local_frames.append(local)
    pr, sr = pearsonr(local['G_width'], local['P']), spearmanr(local['G_width'], local['P'])
    correlation_rows.append({'seed': seed, 'pearson_r': pr.statistic, 'pearson_p': pr.pvalue, 'spearman_rho': sr.statistic, 'spearman_p': sr.pvalue})
local_all = pd.concat(local_frames, ignore_index=True)
seed_correlations = pd.DataFrame(correlation_rows)
pooled_pr = pearsonr(local_all['G_width'], local_all['P'])
pooled_sr = spearmanr(local_all['G_width'], local_all['P'])
seed_correlations.to_csv(RUN_DIR / 'three_seed_correlations.csv', index=False)
display(seed_correlations)
print(f'Pooled Pearson r={pooled_pr.statistic:.4f}, p={pooled_pr.pvalue:.4g}')
print(f'Pooled Spearman rho={pooled_sr.statistic:.4f}, p={pooled_sr.pvalue:.4g}')

## 6. Build the twelve-row confirmatory table

In [ ]:
central = pd.read_csv(RUN_DIR / 'central_analysis_all_seeds.csv')
confirmatory = central.loc[central['specialization_gap_if_available'].notna(), ['seed', 'budget', 'local_wasserstein_sensitivity', 'accuracy', 'specialized_accuracy_if_available', 'specialization_gap_if_available', 'nearest_anchor_distance', 'flops', 'params', 'best_specialization_epoch_if_available', 'best_validation_accuracy_if_available']].copy()
confirmatory.columns = ['seed', 'width', 'G', 'shared_test_accuracy', 'specialized_test_accuracy', 'specialization_gap', 'coverage', 'flops', 'params', 'best_epoch', 'best_validation_accuracy']
confirmatory['width'] = confirmatory['width'].round(2)
confirmatory['best_epoch'] = confirmatory['best_epoch'].astype(int)
confirmatory['log_flops'] = np.log(confirmatory['flops'].astype(float))
confirmatory = confirmatory.sort_values(['seed', 'width']).reset_index(drop=True)
assert len(confirmatory) == 12 and set(confirmatory['width']) == {0.30, 0.40, 0.60, 0.80}
assert np.isfinite(confirmatory.drop(columns=['seed', 'width']).to_numpy(float)).all()
confirmatory.to_csv(RUN_DIR / 'confirmatory_specialization_table.csv', index=False)
display(confirmatory)

## 7. Primary matched 0.40-versus-0.60 test

In [ ]:
matched_rows = []
for seed, frame in confirmatory.groupby('seed'):
    indexed = frame.set_index('width')
    assert np.isclose(indexed.loc[0.40, 'coverage'], indexed.loc[0.60, 'coverage'])
    delta_G = indexed.loc[0.40, 'G'] - indexed.loc[0.60, 'G']
    delta_gap = indexed.loc[0.40, 'specialization_gap'] - indexed.loc[0.60, 'specialization_gap']
    matched_rows.append({'seed': int(seed), 'coverage_0.40': indexed.loc[0.40, 'coverage'], 'coverage_0.60': indexed.loc[0.60, 'coverage'], 'G_0.40': indexed.loc[0.40, 'G'], 'G_0.60': indexed.loc[0.60, 'G'], 'delta_G_0.40_minus_0.60': delta_G, 'gap_0.40': indexed.loc[0.40, 'specialization_gap'], 'gap_0.60': indexed.loc[0.60, 'specialization_gap'], 'delta_gap_0.40_minus_0.60': delta_gap, 'both_hold': bool(delta_G > 0 and delta_gap > 0)})
matched = pd.DataFrame(matched_rows)
matched.to_csv(RUN_DIR / 'matched_040_vs_060_by_seed.csv', index=False)
display(matched)
print('Matched comparison holds in all seeds:', bool(matched['both_hold'].all()))

## 8. Leakage-safe leave-one-seed-out predictor comparison

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
models = {
    'coverage': ['coverage'],
    'log FLOPs': ['log_flops'],
    'geometry': ['G'],
    'coverage + geometry': ['coverage', 'G'],
    'coverage + log FLOPs': ['coverage', 'log_flops'],
    'coverage + log FLOPs + geometry': ['coverage', 'log_flops', 'G'],
}
logo = LeaveOneGroupOut()
summary_rows, fold_rows = [], []
for name, columns in models.items():
    y_true_all, y_pred_all = [], []
    for train_idx, heldout_idx in logo.split(confirmatory, groups=confirmatory['seed']):
        pipeline = make_pipeline(StandardScaler(), LinearRegression())
        pipeline.fit(confirmatory.loc[train_idx, columns], confirmatory.loc[train_idx, 'specialization_gap'])
        predictions = pipeline.predict(confirmatory.loc[heldout_idx, columns])
        truth = confirmatory.loc[heldout_idx, 'specialization_gap'].to_numpy()
        heldout_seed = int(confirmatory.loc[heldout_idx, 'seed'].iloc[0])
        fold_rows.append({'model': name, 'heldout_seed': heldout_seed, 'mae': mean_absolute_error(truth, predictions)})
        y_true_all.extend(truth); y_pred_all.extend(predictions)
    summary_rows.append({'model': name, 'predictors': ' + '.join(columns), 'loso_mae': mean_absolute_error(y_true_all, y_pred_all), 'loso_r2': r2_score(y_true_all, y_pred_all)})
loso = pd.DataFrame(summary_rows).sort_values('loso_mae').reset_index(drop=True)
loso_folds = pd.DataFrame(fold_rows).sort_values(['model', 'heldout_seed']).reset_index(drop=True)
loso.to_csv(RUN_DIR / 'loso_predictor_comparison.csv', index=False)
loso_folds.to_csv(RUN_DIR / 'loso_mae_by_heldout_seed.csv', index=False)
display(loso); display(loso_folds)
print('StandardScaler is inside each fold-specific pipeline: no held-out-seed preprocessing leakage.')

## 9. Decision, plots, report, and archive

In [ ]:
base_mae = float(loso.loc[loso['model'] == 'coverage + log FLOPs', 'loso_mae'].iloc[0])
geometry_mae = float(loso.loc[loso['model'] == 'coverage + log FLOPs + geometry', 'loso_mae'].iloc[0])
matched_pass = bool(matched['both_hold'].all())
incremental_pass = geometry_mae < base_mae
confirmatory_pass = matched_pass and incremental_pass
decision = 'CONFIRMATORY PASS' if confirmatory_pass else 'HYPOTHESIS NOT CONFIRMED'
from IPython.display import Markdown, display
display(Markdown(f'## {decision}'))
print('Matched 0.40/0.60 pass:', matched_pass)
print(f'LOSO base MAE={base_mae:.6f}; with geometry MAE={geometry_mae:.6f}')
print('Geometry improves LOSO MAE:', incremental_pass)

In [ ]:
import matplotlib.pyplot as plt
PLOT_DIR = RUN_DIR / 'plots'; PLOT_DIR.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for seed, frame in confirmatory.groupby('seed'):
    axes[0].plot(frame['width'], frame['G'], marker='o', label=f'seed {seed}')
    axes[1].plot(frame['width'], frame['specialization_gap'], marker='o', label=f'seed {seed}')
axes[0].set(xlabel='Width', ylabel='Pre-specialized G(c)', title='Geometry by specialized width')
axes[1].set(xlabel='Width', ylabel='Test specialization gap', title='Specialization benefit')
for ax in axes: ax.grid(alpha=.25); ax.legend()
fig.tight_layout(); fig.savefig(PLOT_DIR / 'confirmatory_geometry_and_gap.png', dpi=180); plt.show()

In [ ]:
report_lines = [
    '# Confirmatory specialization report', '',
    '- CIFAR-100: fixed 45k train / 5k validation / 10k test',
    '- Fixed non-augmented BN calibration subset: 6,400 samples from 45k train',
    '- Shared: seeds 0,1,2; 20 epochs; anchors 0.25,0.50,0.75,1.00',
    '- Specialized: widths 0.30,0.40,0.60,0.80; 30 epochs; best validation checkpoint',
    '- Test used only after checkpoint selection', '',
    '## Dense G-versus-P correlations', '', seed_correlations.to_markdown(index=False), '',
    f'Pooled Pearson r={pooled_pr.statistic:.6f}; pooled Spearman rho={pooled_sr.statistic:.6f}', '',
    '## Twelve specialization observations', '', confirmatory.to_markdown(index=False), '',
    '## Matched 0.40 versus 0.60', '', matched.to_markdown(index=False), '',
    '## LOSO predictor comparison', '', loso.to_markdown(index=False), '',
    '## LOSO MAE by held-out seed', '', loso_folds.to_markdown(index=False), '',
    f'## Decision: {decision}', '',
    f'Matched pass: {matched_pass}; base MAE={base_mae:.6f}; geometry MAE={geometry_mae:.6f}; incremental pass={incremental_pass}.', '',
    '> With twelve observations and three seed clusters, LOSO is a confirmatory diagnostic rather than a final significance claim.',
]
REPORT = RUN_DIR / 'reports' / 'confirmatory_specialization_report.md'
REPORT.parent.mkdir(parents=True, exist_ok=True)
REPORT.write_text('\n'.join(report_lines) + '\n')
display(Markdown('\n'.join(report_lines)))

In [ ]:
import shutil, zipfile
from IPython.display import FileLink
artifact_files = sorted(path for path in RUN_DIR.rglob('*') if path.is_file())
manifest = pd.DataFrame({'relative_path': [str(p.relative_to(RUN_DIR)) for p in artifact_files], 'size_bytes': [p.stat().st_size for p in artifact_files]})
manifest.to_csv(RUN_DIR / 'artifact_manifest.csv', index=False)
archive = Path(shutil.make_archive(str(Path('/kaggle/working') / RUN_NAME), 'zip', root_dir=RUN_DIR))
with zipfile.ZipFile(archive) as zf: archived = set(zf.namelist())
for required in ['confirmatory_specialization_table.csv', 'matched_040_vs_060_by_seed.csv', 'loso_predictor_comparison.csv', 'loso_mae_by_heldout_seed.csv', 'reports/confirmatory_specialization_report.md']:
    assert required in archived, f'Missing from ZIP: {required}'
print(f'Archived {len(archived)} entries; manifest contains {len(manifest)} files.')
print('Run directory:', RUN_DIR); print('Archive:', archive)
display(FileLink(str(archive)))
print('Use Kaggle Save Version so this multi-hour result persists.')